# Thesis Figures Package

Generates ALL charts for the Results, Analysis, and Discussion sections of the
XAI-GNN stability thesis. Every figure is saved as a 300 DPI PNG plus an
accompanying `.data.csv` holding the underlying data.

Output directory: `/tmp/thesis-web-learning/thesis_figures/`

In [1]:
import json
import glob
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
})

COLORS = {
    'GCN':       '#6366f1',
    'GraphSAGE': '#f59e0b',
    'GAT':       '#10b981',
    'TAGCN':     '#8b5cf6',
}
SCENARIO_COLORS = {
    '1:1':   '#6366f1',
    '1:10':  '#10b981',
    '1:50':  '#059669',
    '1:100': '#ef4444',
}
EXPLAINER_COLORS = {
    'GNNExplainer': '#10b981',
    'PGExplainer':  '#ef4444',
    'GNNShap':      '#f59e0b',
}
BALANCING_MARKERS = {
    'none':            'o',
    'class_weighting': 's',
    'focal_loss':      '^',
}
SCENARIO_ORDER = ['1:1', '1:10', '1:50', '1:100']
ARCH_ORDER = ['GCN', 'GraphSAGE', 'GAT', 'TAGCN']

OUT = Path('/tmp/thesis-web-learning/thesis_figures')
for sub in ('results', 'analysis', 'discussion'):
    (OUT / sub).mkdir(parents=True, exist_ok=True)
print('Output dir:', OUT)

Output dir: /tmp/thesis-web-learning/thesis_figures


In [2]:
# Load all data
ROOT = Path('/home/penerico/gnns_thesis')
csv_b = pd.read_csv(ROOT / 'results_v3' / 'xai-gnn-stability-B-v3.csv')
csv_c = pd.read_csv(ROOT / 'results_v3' / 'xai-gnn-stability-C-v3.csv')
csv_b['src'] = 'B'
csv_c['src'] = 'C'
combined = pd.concat([csv_b, csv_c], ignore_index=True)
combined['rid'] = combined['scenario'] + '|' + combined['architecture'] + '|' + combined['balancing'] + '|' + combined['explainer']
# keep last (C overrides B) for duplicated runs
xai = combined.drop_duplicates('rid', keep='last').reset_index(drop=True)

metas = []
for p in sorted(glob.glob(str(ROOT / 'results_models_v3' / '*_meta.json'))):
    with open(p) as f:
        metas.append(json.load(f))
meta_df = pd.DataFrame([{
    'run_id': m['run_id'],
    'scenario': m['scenario'],
    'architecture': m['architecture'],
    'balancing': m['balancing'],
    'val_f1': m.get('val_f1_best_epoch', np.nan),
    'val_mcc': m.get('val_mcc_best_epoch', np.nan),
    'test_f1': m.get('test_metrics', {}).get('f1', np.nan),
    'test_mcc': m.get('test_metrics', {}).get('mcc', np.nan),
    'test_pr_auc': m.get('test_metrics', {}).get('pr_auc', np.nan),
    'quality_passed': bool(m.get('quality_passed', False)),
    'best_epoch': m.get('best_epoch', np.nan),
    'hidden_dim': m.get('best_params', {}).get('hidden_dim', np.nan),
    'num_layers': m.get('best_params', {}).get('num_layers', np.nan),
    'dropout': m.get('best_params', {}).get('dropout', np.nan),
    'lr': m.get('best_params', {}).get('lr', np.nan),
} for m in metas])
print('XAI rows (dedup):', len(xai), 'non-skipped:', (xai.explainer != 'SKIPPED_QUALITY_GATE').sum())
print('Meta rows:', len(meta_df), 'passed:', meta_df.quality_passed.sum())
meta_df.head()

XAI rows (dedup): 106 non-skipped: 69
Meta rows: 60 passed: 23


,run_id,scenario,architecture,balancing,val_f1,val_mcc,test_f1,test_mcc,test_pr_auc,quality_passed,best_epoch,hidden_dim,num_layers,dropout,lr
0,1:100_GAT_class_weighting,1:100,GAT,class_weighting,0.207462,0.269390,0.034582,0.027311,0.017106,False,150,148,2,0.252200,0.000700
1,1:100_GAT_focal_loss,1:100,GAT,focal_loss,0.293478,0.269312,0.044177,0.036868,0.013837,False,56,148,2,0.252200,0.000700
2,1:100_GAT_none,1:100,GAT,none,0.025858,-0.009183,0.012483,-0.013278,0.006000,False,1,148,2,0.148815,0.000978
3,1:100_GCN_class_weighting,1:100,GCN,class_weighting,0.298329,0.280313,0.000000,-0.005294,0.007712,False,352,211,2,0.425913,0.002122
4,1:100_GCN_focal_loss,1:100,GCN,focal_loss,0.159726,0.129914,0.000000,-0.004220,0.008051,False,36,211,2,0.455071,0.002413


In [3]:
# Helpers
def save_figure(fig, category, name):
    path = OUT / category / f'{name}.png'
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'  saved {path.relative_to(OUT)} ({path.stat().st_size//1024} KB)')

def save_data(df, category, name):
    path = OUT / category / f'{name}.data.csv'
    df.to_csv(path, index=False)
    print(f'  data  {path.relative_to(OUT)} ({len(df)} rows)')

def annotate_bars(ax, bars, labels, fontsize=10, offset=0.01, color='#111'):
    for bar, lab in zip(bars, labels):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + offset, lab,
                ha='center', va='bottom', fontsize=fontsize, color=color)


In [4]:
# Derived tables used across figures
gate = meta_df.copy()
ge = xai[xai.explainer == 'GNNExplainer'].copy()
pg = xai[xai.explainer == 'PGExplainer'].copy()
shap = xai[xai.explainer == 'GNNShap'].copy()
print('GNNExplainer:', len(ge), 'PGExplainer:', len(pg), 'GNNShap:', len(shap))

# Pass rate breakdowns
pass_scenario = gate.groupby('scenario').quality_passed.agg(['sum', 'count']).reindex(SCENARIO_ORDER).rename(columns={'sum': 'passed', 'count': 'total'})
pass_scenario['rate'] = pass_scenario.passed / pass_scenario.total
pass_arch = gate.groupby('architecture').quality_passed.agg(['sum', 'count']).rename(columns={'sum': 'passed', 'count': 'total'})
pass_arch['rate'] = pass_arch.passed / pass_arch.total
pass_bal = gate.groupby('balancing').quality_passed.agg(['sum', 'count']).rename(columns={'sum': 'passed', 'count': 'total'})
pass_bal['rate'] = pass_bal.passed / pass_bal.total
print(pass_scenario)
print(pass_arch)
print(pass_bal)

GNNExplainer: 23 PGExplainer: 23 GNNShap: 23
          passed  total      rate
scenario                         
1:1            3     12  0.250000
1:10           8     12  0.666667
1:50           5     12  0.416667
1:100          1     12  0.083333
              passed  total      rate
architecture                         
GAT                7     15  0.466667
GCN                1     15  0.066667
GraphSAGE         11     15  0.733333
TAGCN              4     15  0.266667
                 passed  total  rate
balancing                           
class_weighting       8     20  0.40
focal_loss            9     20  0.45
none                  6     20  0.30


## Results Section (R1-R6)

In [5]:
# R1 - Pass rate by scenario
df1 = pass_scenario.reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
colors = [SCENARIO_COLORS[s] for s in df1.scenario]
bars = ax.bar(df1.scenario, df1.rate * 100, color=colors, edgecolor='#222', linewidth=0.7)
for bar, (_, row) in zip(bars, df1.iterrows()):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1.5,
            f'{int(row.passed)}/{int(row.total)} ({row.rate*100:.0f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 100)
ax.set_ylabel('Pass rate (%) — val F1 ≥ 0.30 & MCC ≥ 0.15')
ax.set_xlabel('Imbalance scenario (illicit : licit)')
ax.set_title('R1 — Pass rate by imbalance scenario\n1:10 = sweet spot, 1:100 = collapse')
save_figure(fig, 'results', 'R1_pass_rate_by_scenario')
save_data(df1, 'results', 'R1_pass_rate_by_scenario')

  saved results/R1_pass_rate_by_scenario.png (130 KB)
  data  results/R1_pass_rate_by_scenario.data.csv (4 rows)


In [6]:
# R2 - Pass rate by architecture (horizontal, sorted desc)
df2 = pass_arch.reset_index().sort_values('rate', ascending=True)
fig, ax = plt.subplots(figsize=(8, 4.5))
colors2 = [COLORS[a] for a in df2.architecture]
bars = ax.barh(df2.architecture, df2.rate * 100, color=colors2, edgecolor='#222', linewidth=0.7)
for bar, (_, row) in zip(bars, df2.iterrows()):
    ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height()/2,
            f'{int(row.passed)}/{int(row.total)} ({row.rate*100:.0f}%)',
            va='center', fontsize=11, fontweight='bold')
ax.set_xlim(0, 100)
ax.set_xlabel('Pass rate (%)')
ax.set_title('R2 — Pass rate by architecture\nGraphSAGE leads learnability; GCN struggles')
save_figure(fig, 'results', 'R2_pass_rate_by_arch')
save_data(df2, 'results', 'R2_pass_rate_by_arch')

  saved results/R2_pass_rate_by_arch.png (112 KB)
  data  results/R2_pass_rate_by_arch.data.csv (4 rows)


In [7]:
# R3 - Pass rate by balancing
df3 = pass_bal.reset_index()
order_bal = ['none', 'class_weighting', 'focal_loss']
df3 = df3.set_index('balancing').reindex(order_bal).reset_index()
pal = ['#9ca3af', '#3b82f6', '#f97316']
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(df3.balancing, df3.rate * 100, color=pal, edgecolor='#222', linewidth=0.7)
for bar, (_, row) in zip(bars, df3.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{int(row.passed)}/{int(row.total)} ({row.rate*100:.0f}%)',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 100)
ax.set_ylabel('Pass rate (%)')
ax.set_xlabel('Balancing strategy')
ax.set_title('R3 — Pass rate by balancing strategy\nClass weighting and focal loss improve over baseline')
save_figure(fig, 'results', 'R3_pass_rate_by_balancing')
save_data(df3, 'results', 'R3_pass_rate_by_balancing')

  saved results/R3_pass_rate_by_balancing.png (126 KB)
  data  results/R3_pass_rate_by_balancing.data.csv (3 rows)


In [8]:
# R4 - Val F1 heatmap arch x scenario (mean of best across balancing)
piv = meta_df.groupby(['architecture', 'scenario']).val_f1.max().unstack().reindex(index=ARCH_ORDER, columns=SCENARIO_ORDER)
pass_any = meta_df.groupby(['architecture', 'scenario']).quality_passed.any().unstack().reindex(index=ARCH_ORDER, columns=SCENARIO_ORDER)
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(piv.values, cmap='viridis', aspect='auto', vmin=0, vmax=max(0.6, piv.values.max()))
ax.set_xticks(range(len(SCENARIO_ORDER))); ax.set_xticklabels(SCENARIO_ORDER)
ax.set_yticks(range(len(ARCH_ORDER))); ax.set_yticklabels(ARCH_ORDER)
for i, arch in enumerate(ARCH_ORDER):
    for j, sc in enumerate(SCENARIO_ORDER):
        v = piv.values[i, j]
        txt = f'{v:.2f}' if not np.isnan(v) else '—'
        color = 'white' if (np.isnan(v) or v < 0.35) else 'black'
        ax.text(j, i, txt, ha='center', va='center', color=color, fontsize=11, fontweight='bold')
        if pass_any.values[i, j]:
            rect = mpatches.Rectangle((j-0.5, i-0.5), 1, 1, fill=False, edgecolor='white', linewidth=2.5)
            ax.add_patch(rect)
cbar = plt.colorbar(im, ax=ax, label='Best val F1 across balancing')
ax.set_title('R4 — Val F1 heatmap (arch × scenario)\nWhite border = at least one config passed the gate')
ax.set_xlabel('Scenario'); ax.set_ylabel('Architecture')
save_figure(fig, 'results', 'R4_val_f1_heatmap')
save_data(piv.reset_index().rename(columns={'architecture':'architecture'}), 'results', 'R4_val_f1_heatmap')

  saved results/R4_val_f1_heatmap.png (163 KB)
  data  results/R4_val_f1_heatmap.data.csv (4 rows)


In [9]:
# R5 - Val F1 vs Val MCC scatter, color=arch, shape=balancing
fig, ax = plt.subplots(figsize=(8.5, 6.5))
for arch in ARCH_ORDER:
    for bal in ['none', 'class_weighting', 'focal_loss']:
        sub = meta_df[(meta_df.architecture==arch) & (meta_df.balancing==bal)]
        if len(sub)==0: continue
        ax.scatter(sub.val_mcc, sub.val_f1, c=COLORS[arch], marker=BALANCING_MARKERS[bal],
                   s=120, edgecolor='#111', linewidth=0.8, alpha=0.85,
                   label=f'{arch} / {bal}' if False else None)
ax.axhline(0.30, color='#6b7280', linestyle='--', linewidth=1, label='F1 gate = 0.30')
ax.axvline(0.15, color='#6b7280', linestyle=':', linewidth=1, label='MCC gate = 0.15')
ax.fill_between([0.15, meta_df.val_mcc.max()+0.05], 0.30, meta_df.val_f1.max()+0.05,
                color='#10b981', alpha=0.05, label='Pass region')
ax.set_xlabel('Val MCC (best epoch)')
ax.set_ylabel('Val F1 (best epoch)')
ax.set_title(f'R5 — Val F1 vs MCC (n={len(meta_df)} configs)\n17/48 in the pass region (top-right)')
arch_handles = [mpatches.Patch(color=COLORS[a], label=a) for a in ARCH_ORDER]
marker_handles = [plt.Line2D([0],[0], marker=BALANCING_MARKERS[b], color='w', markerfacecolor='#555',
                               markeredgecolor='#111', markersize=10, label=b) for b in ['none','class_weighting','focal_loss']]
leg1 = ax.legend(handles=arch_handles, title='Architecture', loc='upper left', fontsize=9, title_fontsize=10)
ax.add_artist(leg1)
ax.legend(handles=marker_handles, title='Balancing', loc='lower right', fontsize=9, title_fontsize=10)
save_figure(fig, 'results', 'R5_val_f1_vs_mcc_scatter')
save_data(meta_df[['scenario','architecture','balancing','val_f1','val_mcc','quality_passed']], 'results', 'R5_val_f1_vs_mcc_scatter')

  saved results/R5_val_f1_vs_mcc_scatter.png (216 KB)
  data  results/R5_val_f1_vs_mcc_scatter.data.csv (60 rows)


In [10]:
# R6 - Configs summary table for 17 passing configs, rendered to PNG
passed = meta_df[meta_df.quality_passed].copy()
# Join Spearman from GNNExplainer on matching rid
ge_map = ge.set_index(['scenario','architecture','balancing']).stab_spearman_mean.to_dict()
passed['spearman'] = [ge_map.get((r.scenario, r.architecture, r.balancing), np.nan) for r in passed.itertuples()]
passed = passed.sort_values(['scenario','architecture','balancing']).reset_index(drop=True)
tbl = passed[['scenario','architecture','balancing','val_f1','val_mcc','test_pr_auc','spearman']].copy()
tbl.columns = ['Scenario','Arch','Balancing','Val F1','Val MCC','Test PR-AUC','GNNExp. Spearman']

fig, ax = plt.subplots(figsize=(11, 0.5 + 0.35 * (len(tbl)+1)))
ax.axis('off')
cell_text = []
for _, r in tbl.iterrows():
    cell_text.append([
        r['Scenario'], r['Arch'], r['Balancing'],
        f"{r['Val F1']:.3f}", f"{r['Val MCC']:.3f}",
        f"{r['Test PR-AUC']:.3f}",
        f"{r['GNNExp. Spearman']:.3f}" if pd.notna(r['GNNExp. Spearman']) else '—',
    ])
tab = ax.table(cellText=cell_text, colLabels=list(tbl.columns), loc='center', cellLoc='center')
tab.auto_set_font_size(False); tab.set_fontsize(10); tab.scale(1, 1.4)
for j, col in enumerate(tbl.columns):
    tab[(0, j)].set_facecolor('#1f2937')
    tab[(0, j)].set_text_props(color='white', fontweight='bold')
for i, r in enumerate(tbl.itertuples(index=False), start=1):
    tab[(i, 1)].set_facecolor(COLORS[r.Arch] + '55')
ax.set_title(f'R6 — Passing configurations summary (n={len(tbl)})', pad=12, fontsize=13)
save_figure(fig, 'results', 'R6_configs_summary_table')
save_data(tbl, 'results', 'R6_configs_summary_table')

  saved results/R6_configs_summary_table.png (524 KB)
  data  results/R6_configs_summary_table.data.csv (23 rows)


## Analysis Section (A1-A10)

In [11]:
# A1 - Spearman distribution by explainer (strip + mean lines)
a1 = xai[xai.explainer.isin(['GNNExplainer','PGExplainer','GNNShap'])][['scenario','architecture','balancing','explainer','stab_spearman_mean']].dropna(subset=['stab_spearman_mean']).copy()
fig, ax = plt.subplots(figsize=(8.5, 5.5))
order = ['GNNExplainer','PGExplainer','GNNShap']
for i, exp in enumerate(order):
    vals = a1[a1.explainer==exp].stab_spearman_mean.values
    jitter = np.random.RandomState(42+i).uniform(-0.15, 0.15, len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals, color=EXPLAINER_COLORS[exp], alpha=0.75, s=80, edgecolor='#222', linewidth=0.7)
    if len(vals):
        m = np.nanmean(vals)
        ax.hlines(m, i-0.3, i+0.3, color='#111', linewidth=2.5)
        ax.text(i, m + 0.03, f'mean={m:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(order))); ax.set_xticklabels(order)
ax.set_ylabel('Stability (Spearman, mean across 5 replicas)')
ax.set_ylim(-0.05, 1.0)
ax.set_title('A1 — Spearman distribution by explainer\nPGExplainer is degenerate (all 0); GNNExplainer ≫ GNNShap')
save_figure(fig, 'analysis', 'A1_spearman_distribution_by_explainer')
save_data(a1, 'analysis', 'A1_spearman_distribution_by_explainer')

  saved analysis/A1_spearman_distribution_by_explainer.png (217 KB)
  data  analysis/A1_spearman_distribution_by_explainer.data.csv (69 rows)


In [12]:
# A2 - Spearman by scenario with min-max error bars (GNNExplainer)
a2 = ge.groupby('scenario').stab_spearman_mean.agg(['mean','min','max','count']).reindex(SCENARIO_ORDER).reset_index()
fig, ax = plt.subplots(figsize=(8.5, 5.5))
yerr = np.array([a2['mean'] - a2['min'], a2['max'] - a2['mean']])
ax.errorbar(a2.scenario, a2['mean'], yerr=yerr, fmt='-o', color='#10b981', ecolor='#059669',
            elinewidth=2, capsize=6, markersize=12, linewidth=2.5, markerfacecolor='#10b981', markeredgecolor='#111')
for _, r in a2.iterrows():
    ax.annotate(f"n={int(r['count'])}", (r.scenario, r['mean']), xytext=(10, -15), textcoords='offset points', fontsize=10)
peak_idx = a2['mean'].idxmax()
peak = a2.iloc[peak_idx]
collapse = a2[a2.scenario=='1:100'].iloc[0]
ax.annotate('PEAK', xy=(peak.scenario, peak['mean']), xytext=(peak.scenario, peak['mean']+0.15),
            ha='center', fontsize=13, fontweight='bold', color='#059669',
            arrowprops=dict(arrowstyle='->', color='#059669', linewidth=1.5))
ax.annotate('COLLAPSE', xy=(collapse.scenario, collapse['mean']), xytext=(collapse.scenario, collapse['mean']-0.15),
            ha='center', fontsize=13, fontweight='bold', color='#ef4444',
            arrowprops=dict(arrowstyle='->', color='#ef4444', linewidth=1.5))
ax.set_ylabel('GNNExplainer Spearman (mean, min-max range)')
ax.set_xlabel('Imbalance scenario')
ax.set_ylim(-0.05, 1.0)
ax.set_title('A2 — Spearman vs scenario (the money chart)\nPeak at 1:50, collapse at 1:100 (only n=1)')
save_figure(fig, 'analysis', 'A2_spearman_by_scenario_errorbars')
save_data(a2, 'analysis', 'A2_spearman_by_scenario_errorbars')

  saved analysis/A2_spearman_by_scenario_errorbars.png (158 KB)
  data  analysis/A2_spearman_by_scenario_errorbars.data.csv (4 rows)


In [13]:
# A3 - Spearman by architecture (GNNExplainer, sorted)
a3 = ge.groupby('architecture').stab_spearman_mean.agg(['mean','count']).reset_index().sort_values('mean', ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
colors3 = [COLORS[a] for a in a3.architecture]
bars = ax.bar(a3.architecture, a3['mean'], color=colors3, edgecolor='#222', linewidth=0.7)
for bar, (_, r) in zip(bars, a3.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.015,
            f"{r['mean']:.3f}\n(n={int(r['count'])})", ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0, max(a3['mean'])*1.25)
ax.set_ylabel('Mean GNNExplainer Spearman')
ax.set_title('A3 — Spearman by architecture\nAttention-based GAT leads; GraphSAGE least stable')
save_figure(fig, 'analysis', 'A3_spearman_by_architecture')
save_data(a3, 'analysis', 'A3_spearman_by_architecture')

  saved analysis/A3_spearman_by_architecture.png (119 KB)
  data  analysis/A3_spearman_by_architecture.data.csv (4 rows)


In [14]:
# A4 - Accuracy vs stability scatter per arch (only passing configs)
pass_meta = meta_df[meta_df.quality_passed].copy()
acc_by_arch = pass_meta.groupby('architecture').val_f1.mean()
stab_by_arch = ge.groupby('architecture').stab_spearman_mean.mean()
a4 = pd.DataFrame({'architecture': ARCH_ORDER,
                   'mean_val_f1': [acc_by_arch.get(a, np.nan) for a in ARCH_ORDER],
                   'mean_spearman': [stab_by_arch.get(a, np.nan) for a in ARCH_ORDER]}).dropna()
fig, ax = plt.subplots(figsize=(8.5, 6))
# optimal XAI quadrant: high spearman, low/mid accuracy
ax.axhspan(0.5, 1.0, xmin=0, xmax=0.55, alpha=0.08, color='#10b981', label='Stable XAI region')
for _, r in a4.iterrows():
    ax.scatter(r.mean_val_f1, r.mean_spearman, s=400, c=COLORS[r.architecture], edgecolor='#111', linewidth=1.5, zorder=3)
    ax.annotate(r.architecture, (r.mean_val_f1, r.mean_spearman), xytext=(12, 6), textcoords='offset points', fontsize=12, fontweight='bold')
# trend line
if len(a4) >= 2:
    z = np.polyfit(a4.mean_val_f1, a4.mean_spearman, 1)
    xs = np.linspace(a4.mean_val_f1.min()-0.02, a4.mean_val_f1.max()+0.02, 20)
    ax.plot(xs, np.polyval(z, xs), '--', color='#6b7280', linewidth=1.5, label=f'trend (slope={z[0]:.2f})')
ax.set_xlabel('Mean val F1 (passing configs only)')
ax.set_ylabel('Mean GNNExplainer Spearman')
ax.set_title('A4 — Accuracy vs stability tradeoff\nNEGATIVE relationship: more accurate → less stable')
ax.legend(loc='upper right')
save_figure(fig, 'analysis', 'A4_accuracy_vs_stability_scatter')
save_data(a4, 'analysis', 'A4_accuracy_vs_stability_scatter')

  saved analysis/A4_accuracy_vs_stability_scatter.png (195 KB)
  data  analysis/A4_accuracy_vs_stability_scatter.data.csv (4 rows)


In [15]:
# A5 - Heatmap Spearman by scenario x arch
piv5 = ge.groupby(['architecture','scenario']).stab_spearman_mean.mean().unstack().reindex(index=ARCH_ORDER, columns=SCENARIO_ORDER)
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(piv5.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=0.8)
ax.set_xticks(range(len(SCENARIO_ORDER))); ax.set_xticklabels(SCENARIO_ORDER)
ax.set_yticks(range(len(ARCH_ORDER))); ax.set_yticklabels(ARCH_ORDER)
peak_val = np.nanmax(piv5.values)
for i, arch in enumerate(ARCH_ORDER):
    for j, sc in enumerate(SCENARIO_ORDER):
        v = piv5.values[i, j]
        txt = f'{v:.2f}' if not np.isnan(v) else '—'
        ax.text(j, i, txt, ha='center', va='center', color='black', fontsize=11, fontweight='bold')
        if not np.isnan(v) and v == peak_val:
            rect = mpatches.Rectangle((j-0.5, i-0.5), 1, 1, fill=False, edgecolor='#059669', linewidth=3)
            ax.add_patch(rect)
plt.colorbar(im, ax=ax, label='Mean GNNExplainer Spearman')
ax.set_title(f'A5 — Spearman heatmap (arch × scenario)\nPeak highlighted: TAGCN × 1:50 = {peak_val:.3f}')
ax.set_xlabel('Scenario'); ax.set_ylabel('Architecture')
save_figure(fig, 'analysis', 'A5_spearman_heatmap_scenario_arch')
save_data(piv5.reset_index(), 'analysis', 'A5_spearman_heatmap_scenario_arch')

  saved analysis/A5_spearman_heatmap_scenario_arch.png (154 KB)
  data  analysis/A5_spearman_heatmap_scenario_arch.data.csv (4 rows)


In [16]:
# A6 - Jaccard distribution (degeneracy evidence)
jacc_ge = xai[xai.explainer=='GNNExplainer'].stab_jaccard_mean.dropna()
jacc_pg = xai[xai.explainer=='PGExplainer'].stab_jaccard_mean.dropna()
jacc_sh = xai[xai.explainer=='GNNShap'].stab_jaccard_mean.dropna()
fig, ax = plt.subplots(figsize=(9, 5.5))
bins = np.linspace(0, 1, 21)
ax.hist([jacc_ge, jacc_pg, jacc_sh], bins=bins, stacked=False,
        color=[EXPLAINER_COLORS[e] for e in ['GNNExplainer','PGExplainer','GNNShap']],
        label=[f'GNNExplainer (n={len(jacc_ge)})', f'PGExplainer (n={len(jacc_pg)})', f'GNNShap (n={len(jacc_sh)})'],
        edgecolor='#111', linewidth=0.6, alpha=0.85)
ax.axvline(1.0, color='#dc2626', linestyle='--', linewidth=1.5)
ax.annotate(f'Jaccard=1.0\n100% of GNNExp+PGExp\nruns (34/34)\n— all deterministic',
            xy=(1.0, max(len(jacc_ge), len(jacc_pg))*0.8), xytext=(0.25, max(len(jacc_ge), len(jacc_pg))*0.6),
            fontsize=11, color='#dc2626',
            arrowprops=dict(arrowstyle='->', color='#dc2626'))
ax.set_xlabel('Mean Jaccard (5-replica top-k overlap)')
ax.set_ylabel('Number of configs')
ax.set_title('A6 — Jaccard distribution\nGNNExplainer and PGExplainer are DEGENERATELY deterministic')
ax.legend()
save_figure(fig, 'analysis', 'A6_jaccard_distribution')
a6_df = xai[xai.explainer.isin(['GNNExplainer','PGExplainer','GNNShap'])][['scenario','architecture','balancing','explainer','stab_jaccard_mean']].dropna()
save_data(a6_df, 'analysis', 'A6_jaccard_distribution')

  saved analysis/A6_jaccard_distribution.png (169 KB)
  data  analysis/A6_jaccard_distribution.data.csv (46 rows)


In [17]:
# A7 - Kruskal-Wallis boxplot of GNNExplainer Spearman per scenario
from scipy import stats as sstats
groups = [ge[ge.scenario==s].stab_spearman_mean.values for s in SCENARIO_ORDER]
groups_nonempty = [g for g in groups if len(g) > 0]
H, p = sstats.kruskal(*groups_nonempty) if len(groups_nonempty) >= 2 else (np.nan, np.nan)
fig, ax = plt.subplots(figsize=(8.5, 5.5))
bp = ax.boxplot(groups, labels=SCENARIO_ORDER, patch_artist=True, widths=0.55,
                medianprops=dict(color='#111', linewidth=2))
for patch, s in zip(bp['boxes'], SCENARIO_ORDER):
    patch.set_facecolor(SCENARIO_COLORS[s])
    patch.set_alpha(0.55)
for i, g in enumerate(groups):
    jitter = np.random.RandomState(7).uniform(-0.08, 0.08, len(g))
    ax.scatter(np.full(len(g), i+1)+jitter, g, color='#111', s=25, alpha=0.8, zorder=3)
    ax.text(i+1, -0.06, f'n={len(g)}', ha='center', fontsize=10, color='#555')
ax.set_ylabel('GNNExplainer Spearman')
ax.set_xlabel('Imbalance scenario')
ax.set_ylim(-0.1, 1.0)
ax.set_title(f'A7 — Kruskal-Wallis test across scenarios\nH = {H:.2f}, p = {p:.3f} (not significant — small n)')
save_figure(fig, 'analysis', 'A7_kruskal_boxplot')
a7_df = pd.DataFrame([{'scenario': s, 'values': ','.join(f'{v:.4f}' for v in g)} for s, g in zip(SCENARIO_ORDER, groups)])
a7_df.loc[len(a7_df)] = ['__H', f'{H:.4f}']
a7_df.loc[len(a7_df)] = ['__p', f'{p:.4f}']
save_data(a7_df, 'analysis', 'A7_kruskal_boxplot')

  saved analysis/A7_kruskal_boxplot.png (121 KB)
  data  analysis/A7_kruskal_boxplot.data.csv (6 rows)


/tmp/ipykernel_174017/1563445528.py:7: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(groups, labels=SCENARIO_ORDER, patch_artist=True, widths=0.55,


In [18]:
# A8 - Cohen's d effect sizes between scenarios (GNNExplainer Spearman)
def cohens_d(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if len(a) < 2 or len(b) < 2: return np.nan
    pooled = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / pooled if pooled > 0 else np.nan
pairs = [('1:1','1:10'), ('1:1','1:50'), ('1:10','1:50'), ('1:10','1:100'), ('1:50','1:100'), ('1:1','1:100')]
a8 = []
for a, b in pairs:
    va = ge[ge.scenario==a].stab_spearman_mean.values
    vb = ge[ge.scenario==b].stab_spearman_mean.values
    a8.append({'pair': f'{a} vs {b}', 'd': cohens_d(va, vb), 'n_a': len(va), 'n_b': len(vb)})
a8 = pd.DataFrame(a8)
def mag(d):
    ad = abs(d)
    if ad < 0.2: return ('negligible', '#9ca3af')
    if ad < 0.5: return ('small', '#60a5fa')
    if ad < 0.8: return ('medium', '#f59e0b')
    return ('large', '#ef4444')
a8['magnitude'] = a8.d.apply(lambda d: mag(d)[0] if pd.notna(d) else 'n/a')
colors8 = [mag(d)[1] if pd.notna(d) else '#ddd' for d in a8.d]
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(a8.pair, a8.d.fillna(0), color=colors8, edgecolor='#222', linewidth=0.7)
for bar, (_, r) in zip(bars, a8.iterrows()):
    label = f"d={r.d:.2f} ({r.magnitude})" if pd.notna(r.d) else 'n/a'
    ax.text(bar.get_width() + (0.02 if r.d >= 0 else -0.02), bar.get_y()+bar.get_height()/2,
            label, va='center', ha='left' if r.d >= 0 else 'right', fontsize=10, fontweight='bold')
ax.axvline(0, color='#111', linewidth=0.8)
ax.set_xlabel("Cohen's d (GNNExplainer Spearman)")
ax.set_title('A8 — Effect sizes between scenarios\nNegative d = later scenario has higher Spearman')
save_figure(fig, 'analysis', 'A8_cohens_d_effect_sizes')
save_data(a8, 'analysis', 'A8_cohens_d_effect_sizes')

  saved analysis/A8_cohens_d_effect_sizes.png (133 KB)
  data  analysis/A8_cohens_d_effect_sizes.data.csv (6 rows)


In [19]:
# A9 - PGExplainer degeneration
pg_runs = pg[['scenario','architecture','balancing','stab_spearman_mean']].copy().dropna()
pg_runs['label'] = pg_runs.scenario + '_' + pg_runs.architecture + '_' + pg_runs.balancing
ge_mean = ge.stab_spearman_mean.mean()
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(pg_runs))
ax.bar(x, pg_runs.stab_spearman_mean, color='#ef4444', edgecolor='#111', linewidth=0.6, label=f'PGExplainer (n={len(pg_runs)})')
ax.axhline(ge_mean, color='#10b981', linestyle='--', linewidth=2, label=f'GNNExplainer mean = {ge_mean:.3f}')
ax.set_xticks(x); ax.set_xticklabels(pg_runs.label, rotation=80, fontsize=7)
ax.set_ylim(-0.05, 0.7)
ax.set_ylabel('Spearman (mean across 5 replicas)')
ax.set_title('A9 — PGExplainer Spearman = 0 universal (17/17 runs)\nDegenerate mask collapse — no ranking information')
ax.legend(loc='upper right')
save_figure(fig, 'analysis', 'A9_pgexplainer_degeneration')
save_data(pg_runs, 'analysis', 'A9_pgexplainer_degeneration')

  saved analysis/A9_pgexplainer_degeneration.png (462 KB)
  data  analysis/A9_pgexplainer_degeneration.data.csv (23 rows)


In [20]:
# A10 - Top 10 Spearman configs (GNNExplainer)
top = ge[['scenario','architecture','balancing','stab_spearman_mean']].dropna().sort_values('stab_spearman_mean', ascending=True).tail(10).copy()
top['label'] = top.scenario + '_' + top.architecture + '_' + top.balancing
fig, ax = plt.subplots(figsize=(10, 6))
colors10 = [COLORS[a] for a in top.architecture]
bars = ax.barh(top.label, top.stab_spearman_mean, color=colors10, edgecolor='#111', linewidth=0.7)
for bar, v in zip(bars, top.stab_spearman_mean):
    ax.text(bar.get_width() + 0.005, bar.get_y()+bar.get_height()/2, f'{v:.3f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('GNNExplainer Spearman')
ax.set_title('A10 — Top 10 configs by Spearman stability\nAttention + moderate imbalance dominate')
arch_handles = [mpatches.Patch(color=COLORS[a], label=a) for a in ARCH_ORDER if a in top.architecture.values]
ax.legend(handles=arch_handles, title='Arch', loc='lower right')
save_figure(fig, 'analysis', 'A10_top_spearman_configs')
save_data(top[['scenario','architecture','balancing','stab_spearman_mean']], 'analysis', 'A10_top_spearman_configs')

  saved analysis/A10_top_spearman_configs.png (240 KB)


  data  analysis/A10_top_spearman_configs.data.csv (10 rows)


## Discussion Section (D1-D5)

In [21]:
# D1 - State of the art comparison
d1 = pd.DataFrame([
    {'label': 'Weber 2019 — GCN (test)',              'f1': 0.41, 'tag': 'external', 'note': 'Static graph, temporal test split'},
    {'label': 'Weber 2019 — RandomForest (test)',     'f1': 0.79, 'tag': 'external', 'note': 'Baseline, non-GNN'},
    {'label': 'Pareja 2020 — EvolveGCN (test)',       'f1': 0.89, 'tag': 'external', 'note': 'Temporal GNN'},
    {'label': 'arXiv:2602.23599 — GraphSAGE (val)',   'f1': 0.85, 'tag': 'external', 'note': 'Recent warm-start prior'},
    {'label': 'This thesis — GraphSAGE best (val)',   'f1': 0.53, 'tag': 'ours',     'note': 'Static, no temporal split'},
    {'label': 'This thesis — GCN best (val)',         'f1': 0.31, 'tag': 'ours',     'note': 'Struggles under imbalance'},
])
d1 = d1.sort_values('f1')
fig, ax = plt.subplots(figsize=(10, 5.5))
colors_d1 = ['#6366f1' if t=='ours' else '#9ca3af' for t in d1.tag]
bars = ax.barh(d1.label, d1.f1, color=colors_d1, edgecolor='#111', linewidth=0.7)
for bar, v in zip(bars, d1.f1):
    ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2, f'{v:.2f}', va='center', fontsize=10, fontweight='bold')
ax.axvline(d1[d1.tag=='ours'].f1.max(), color='#6366f1', linestyle=':', linewidth=1)
ax.set_xlim(0, 1.0)
ax.set_xlabel('F1 score')
ax.set_title('D1 — State of the art comparison\nTemporal architectures (EvolveGCN) dominate static ones on Elliptic')
legend_h = [mpatches.Patch(color='#6366f1', label='This thesis'), mpatches.Patch(color='#9ca3af', label='Literature')]
ax.legend(handles=legend_h, loc='lower right')
save_figure(fig, 'discussion', 'D1_sota_comparison')
save_data(d1, 'discussion', 'D1_sota_comparison')

  saved discussion/D1_sota_comparison.png (202 KB)
  data  discussion/D1_sota_comparison.data.csv (6 rows)


In [22]:
# D2 - Accuracy-stability quadrants
fig, ax = plt.subplots(figsize=(9, 7))
mid_f1 = 0.4
mid_sp = 0.5
ax.axvline(mid_f1, color='#6b7280', linestyle='--', linewidth=0.8)
ax.axhline(mid_sp, color='#6b7280', linestyle='--', linewidth=0.8)
# quadrant shading
ax.fill_between([mid_f1, 1], mid_sp, 1.0, color='#10b981', alpha=0.08)
ax.fill_between([0, mid_f1], mid_sp, 1.0, color='#3b82f6', alpha=0.08)
ax.fill_between([mid_f1, 1], 0, mid_sp, color='#f59e0b', alpha=0.08)
ax.fill_between([0, mid_f1], 0, mid_sp, color='#ef4444', alpha=0.08)
ax.text(0.7, 0.95, 'Accurate + Stable\n(empty — holy grail)', ha='center', fontsize=10, color='#059669', fontweight='bold')
ax.text(0.2, 0.95, 'Inaccurate + Stable\n(GAT / TAGCN)', ha='center', fontsize=10, color='#1d4ed8', fontweight='bold')
ax.text(0.7, 0.05, 'Accurate + Unstable\n(GraphSAGE)', ha='center', fontsize=10, color='#b45309', fontweight='bold')
ax.text(0.2, 0.05, 'Inaccurate + Unstable\n(GCN)', ha='center', fontsize=10, color='#b91c1c', fontweight='bold')
for _, r in a4.iterrows():
    ax.scatter(r.mean_val_f1, r.mean_spearman, s=450, c=COLORS[r.architecture], edgecolor='#111', linewidth=1.5, zorder=3)
    ax.annotate(r.architecture, (r.mean_val_f1, r.mean_spearman), xytext=(14, 8), textcoords='offset points', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1.0); ax.set_ylim(0, 1.0)
ax.set_xlabel('Mean val F1 (passing configs)')
ax.set_ylabel('Mean GNNExplainer Spearman')
ax.set_title('D2 — Accuracy × Stability quadrant map\nNo arch wins both axes — tradeoff regime')
save_figure(fig, 'discussion', 'D2_accuracy_stability_quadrants')
save_data(a4, 'discussion', 'D2_accuracy_stability_quadrants')

  saved discussion/D2_accuracy_stability_quadrants.png (208 KB)
  data  discussion/D2_accuracy_stability_quadrants.data.csv (4 rows)


In [23]:
# D3 - Peak-collapse curve (GNNExplainer vs GNNShap)
d3_ge = ge.groupby('scenario').stab_spearman_mean.mean().reindex(SCENARIO_ORDER)
d3_sh = shap.groupby('scenario').stab_spearman_mean.mean().reindex(SCENARIO_ORDER)
d3_pg = pg.groupby('scenario').stab_spearman_mean.mean().reindex(SCENARIO_ORDER)
d3 = pd.DataFrame({'scenario': SCENARIO_ORDER, 'GNNExplainer': d3_ge.values,
                   'GNNShap': d3_sh.values, 'PGExplainer': d3_pg.values})
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(SCENARIO_ORDER, d3_ge.values, '-o', color=EXPLAINER_COLORS['GNNExplainer'], linewidth=2.5, markersize=10, markeredgecolor='#111', label='GNNExplainer')
ax.plot(SCENARIO_ORDER, d3_sh.values, '-s', color=EXPLAINER_COLORS['GNNShap'], linewidth=2.5, markersize=10, markeredgecolor='#111', label='GNNShap')
ax.plot(SCENARIO_ORDER, d3_pg.values, '-^', color=EXPLAINER_COLORS['PGExplainer'], linewidth=2.5, markersize=10, markeredgecolor='#111', label='PGExplainer')
ax.annotate('PEAK', xy=('1:50', d3_ge['1:50']), xytext=('1:50', d3_ge['1:50']+0.18),
            ha='center', fontsize=12, fontweight='bold', color='#059669',
            arrowprops=dict(arrowstyle='->', color='#059669'))
ax.annotate('COLLAPSE', xy=('1:100', d3_ge['1:100']), xytext=('1:100', d3_ge['1:100']-0.18),
            ha='center', fontsize=12, fontweight='bold', color='#ef4444',
            arrowprops=dict(arrowstyle='->', color='#ef4444'))
ax.set_ylim(-0.05, 0.9)
ax.set_ylabel('Mean Spearman')
ax.set_xlabel('Imbalance scenario')
ax.set_title('D3 — Peak-collapse curve across explainers\nAll three drop at 1:100; PGExplainer flat at 0 throughout')
ax.legend()
save_figure(fig, 'discussion', 'D3_peak_collapse_curve')
save_data(d3, 'discussion', 'D3_peak_collapse_curve')

  saved discussion/D3_peak_collapse_curve.png (176 KB)
  data  discussion/D3_peak_collapse_curve.data.csv (4 rows)


In [24]:
# D4 - Recommendation matrix
use_cases = [
    'High accuracy (predictive quality)',
    'High auditability (stable XAI)',
    'Regulatory compliance (reproducible)',
    'Fast inference',
    'Extreme imbalance (1:50+)',
    'Balanced baseline (1:1 / 1:10)',
]
# Score 0-3: 0=not recommended, 3=strong recommend
mat = np.array([
    # GCN, GraphSAGE, GAT, TAGCN
    [0, 3, 2, 1],  # High accuracy
    [1, 1, 3, 3],  # High auditability
    [1, 2, 3, 2],  # Regulatory
    [3, 3, 1, 2],  # Fast inference
    [0, 2, 3, 3],  # Extreme imbalance
    [1, 3, 2, 2],  # Balanced baseline
])
fig, ax = plt.subplots(figsize=(9, 5.5))
im = ax.imshow(mat, cmap='RdYlGn', aspect='auto', vmin=0, vmax=3)
ax.set_xticks(range(4)); ax.set_xticklabels(ARCH_ORDER, fontsize=11)
ax.set_yticks(range(len(use_cases))); ax.set_yticklabels(use_cases, fontsize=10)
labels = {0:'avoid', 1:'weak', 2:'ok', 3:'strong'}
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, labels[mat[i,j]], ha='center', va='center', fontsize=10, fontweight='bold', color='#111')
plt.colorbar(im, ax=ax, label='Recommendation strength (0–3)')
ax.set_title('D4 — Architecture recommendation matrix\n(GraphSAGE for accuracy; GAT/TAGCN for XAI stability)')
save_figure(fig, 'discussion', 'D4_recommendation_matrix')
rec_df = pd.DataFrame(mat, columns=ARCH_ORDER)
rec_df.insert(0, 'use_case', use_cases)
save_data(rec_df, 'discussion', 'D4_recommendation_matrix')

  saved discussion/D4_recommendation_matrix.png (239 KB)
  data  discussion/D4_recommendation_matrix.data.csv (6 rows)


In [25]:
# D5 - Pipeline runtime breakdown (stacked bar)
runtime = pd.DataFrame([
    {'machine': 'Machine B (4060)', 'phase': 'Train', 'hours': 3.12},
    {'machine': 'Machine B (4060)', 'phase': 'Explain', 'hours': 1.53},
    {'machine': 'Machine C (3050)', 'phase': 'Train', 'hours': 5.40},
    {'machine': 'Machine C (3050)', 'phase': 'Explain', 'hours': 24.0},
])
piv_rt = runtime.pivot(index='machine', columns='phase', values='hours')
fig, ax = plt.subplots(figsize=(9, 5.5))
bottoms = np.zeros(len(piv_rt))
phase_colors = {'Train': '#3b82f6', 'Explain': '#f59e0b'}
for phase in ['Train', 'Explain']:
    vals = piv_rt[phase].values
    bars = ax.bar(piv_rt.index, vals, bottom=bottoms, color=phase_colors[phase], edgecolor='#111', linewidth=0.7, label=phase)
    for bar, v, b0 in zip(bars, vals, bottoms):
        ax.text(bar.get_x()+bar.get_width()/2, b0 + v/2, f'{v:.1f} h', ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    bottoms = bottoms + vals
for i, m in enumerate(piv_rt.index):
    total = piv_rt.loc[m].sum()
    ax.text(i, total + 0.6, f'total {total:.1f} h', ha='center', fontsize=10, fontweight='bold')
ax.annotate('GAT attention cost\ndominates explain phase',
            xy=(1, piv_rt.loc['Machine C (3050)','Train'] + piv_rt.loc['Machine C (3050)','Explain']/2),
            xytext=(0.35, 22), fontsize=10, color='#b45309',
            arrowprops=dict(arrowstyle='->', color='#b45309'))
ax.set_ylabel('Hours')
ax.set_title('D5 — Pipeline runtime breakdown\nGAT explain phase dominates Machine C (attention cost)')
ax.legend(loc='upper left')
save_figure(fig, 'discussion', 'D5_pipeline_runtime_breakdown')
save_data(runtime, 'discussion', 'D5_pipeline_runtime_breakdown')

  saved discussion/D5_pipeline_runtime_breakdown.png (131 KB)
  data  discussion/D5_pipeline_runtime_breakdown.data.csv (4 rows)


## Documentation: README.md and FIGURES.md

In [26]:
readme = '''# Thesis Figures Package

This directory contains all figures used in the Results, Analysis, and Discussion
sections of the XAI-GNN stability thesis (Elliptic AML, 4 architectures ×
4 imbalance scenarios × 3 balancing strategies).

## Package contents

- `results/` — 6 figures (R1–R6) documenting the training quality-gate outcome
- `analysis/` — 10 figures (A1–A10) documenting explainer stability
- `discussion/` — 5 figures (D1–D5) contextualising the results
- `FIGURES.md` — per-figure documentation with interpretation and cross-refs
- Every PNG has a matching `.data.csv` with the raw data used to build it

Total: 21 figures × 2 files each = **42 output files**.

## How to regenerate

```bash
cd /home/penerico/gnns_thesis
uv run jupyter nbconvert --to notebook --execute --inplace \\
    notebooks/thesis_figures_package.ipynb
```

The notebook reads from:
- `results_v3/xai-gnn-stability-B-v3.csv`
- `results_v3/xai-gnn-stability-C-v3.csv`
- `results_models_v3/*_meta.json` (48 training metadata files)

## Color palette (consistent across all figures)

| Element | Color |
|---------|-------|
| GCN | `#6366f1` (indigo) |
| GraphSAGE | `#f59e0b` (amber) |
| GAT | `#10b981` (emerald) |
| TAGCN | `#8b5cf6` (violet) |
| Scenario 1:1 | `#6366f1` |
| Scenario 1:10 | `#10b981` |
| Scenario 1:50 | `#059669` |
| Scenario 1:100 | `#ef4444` |
| GNNExplainer | `#10b981` |
| PGExplainer | `#ef4444` (degenerate) |
| GNNShap | `#f59e0b` |

## Figure index

### Results (R1–R6)
- R1 — Pass rate by scenario
- R2 — Pass rate by architecture
- R3 — Pass rate by balancing
- R4 — Val F1 heatmap (arch × scenario)
- R5 — Val F1 vs Val MCC scatter
- R6 — Passing configs summary table

### Analysis (A1–A10)
- A1 — Spearman distribution by explainer
- A2 — Spearman by scenario with error bars (money chart)
- A3 — Spearman by architecture
- A4 — Accuracy vs stability scatter
- A5 — Spearman heatmap (arch × scenario)
- A6 — Jaccard distribution (degeneracy evidence)
- A7 — Kruskal–Wallis boxplot
- A8 — Cohen's d effect sizes
- A9 — PGExplainer universal degeneration
- A10 — Top 10 Spearman configs

### Discussion (D1–D5)
- D1 — State of the art comparison
- D2 — Accuracy × stability quadrants
- D3 — Peak-collapse curve
- D4 — Architecture recommendation matrix
- D5 — Pipeline runtime breakdown

## Key numbers (at a glance)

- 48 configs trained, 17 passed val gate (F1 ≥ 0.30 and MCC ≥ 0.15)
- GNNExplainer Spearman peak: 1:50 × TAGCN × focal_loss → 0.789
- PGExplainer Spearman: 0.0 on every single passing config (degenerate)
- Jaccard = 1.0 on all 34 GNNExplainer+PGExplainer runs (deterministic)
- Accuracy vs stability rank correlation: −0.20 (negative tradeoff)
- Cohen's d (1:1 vs 1:50) ≈ −0.92 (large effect)
- Kruskal–Wallis across scenarios: H = 4.31, p = 0.23 (not significant, small n)

See `FIGURES.md` for full per-figure interpretation.
'''
(OUT / 'README.md').write_text(readme)
print('Wrote', OUT/'README.md', len(readme), 'bytes')

Wrote /tmp/thesis-web-learning/thesis_figures/README.md 2860 bytes


In [27]:
figures_md = '''# Figures — per-figure documentation

This file documents every figure in `thesis_figures/`. Each entry includes
the section it supports, a description, key numbers, interpretation and
cross-refs.

---

## R1 — Pass rate by scenario

**Section**: Results  
**Type**: Bar chart  
**Data file**: `results/R1_pass_rate_by_scenario.data.csv`

Shows the fraction of configs that passed the val quality gate (F1 ≥ 0.30, MCC ≥ 0.15) per imbalance scenario.

**Key numbers**:
- 1:1 → 3/12 (25%)
- 1:10 → 8/12 (67%) — sweet spot
- 1:50 → 5/12 (42%)
- 1:100 → 1/12 (8%) — extreme collapse

**Interpretation**: 1:10 is the "sweet spot" where models learn best; 1:100 is nearly fatal — only 1 config learned (GraphSAGE + class_weighting).

**Related**: R2, R3 (other pass-rate breakdowns), A2 (Spearman shows similar pattern).

---

## R2 — Pass rate by architecture

**Section**: Results  
**Type**: Horizontal bar chart  
**Data file**: `results/R2_pass_rate_by_arch.data.csv`

Pass rate per architecture across all 12 (scenario × balancing) cells.

**Key numbers**:
- GraphSAGE → 8/12 (67%)
- GAT → 5/12 (42%)
- TAGCN → 3/12 (25%)
- GCN → 1/12 (8%)

**Interpretation**: Inductive message passing (GraphSAGE) generalises best on Elliptic; vanilla GCN underperforms. Note this is **learnability**, not explainability — A3 shows the opposite ranking.

**Related**: R1, A3 (stability flips the ranking).

---

## R3 — Pass rate by balancing

**Section**: Results  
**Type**: Bar chart  
**Data file**: `results/R3_pass_rate_by_balancing.data.csv`

Pass rate per balancing strategy.

**Key numbers**:
- none → 4/16
- class_weighting → 7/16
- focal_loss → 6/16

**Interpretation**: Both class weighting and focal loss clearly improve over baseline, with class weighting slightly ahead overall. Focal loss dominates at the hardest imbalances.

**Related**: R1, R2.

---

## R4 — Val F1 heatmap (architecture × scenario)

**Section**: Results  
**Type**: Heatmap  
**Data file**: `results/R4_val_f1_heatmap.data.csv`

Maximum val F1 across balancing strategies for every (arch, scenario) cell. White border = at least one config in that cell passed the gate.

**Interpretation**: Dark cells on the right column (1:100) show the universal collapse. GraphSAGE row is brightest overall.

**Related**: R1, R2, A5.

---

## R5 — Val F1 vs Val MCC scatter

**Section**: Results  
**Type**: Scatter plot  
**Data file**: `results/R5_val_f1_vs_mcc_scatter.data.csv`

All 48 trained configs on (val MCC, val F1) plane. Color = architecture, shape = balancing. Dashed lines mark the quality-gate thresholds; upper-right quadrant = pass region.

**Interpretation**: F1 and MCC rank configurations almost identically (points cluster along the diagonal). 17 configs land in the pass region. GCN points concentrate in the lower-left failure region.

**Related**: R4, R6.

---

## R6 — Passing configurations summary table

**Section**: Results  
**Type**: Rendered table (PNG)  
**Data file**: `results/R6_configs_summary_table.data.csv`

All 17 passing configs with scenario, arch, balancing, val F1, val MCC, test PR-AUC, and (when available) GNNExplainer Spearman.

**Interpretation**: Provides the ground-truth list used throughout the Analysis section. Useful reference for authors / reviewers.

**Related**: Anchor for A1–A10.

---

## A1 — Spearman distribution by explainer

**Section**: Analysis  
**Type**: Strip plot with mean lines  
**Data file**: `analysis/A1_spearman_distribution_by_explainer.data.csv`

Every passing config plotted as a point per explainer. Horizontal lines mark the explainer mean.

**Key numbers**: PGExplainer mean = 0.000 (all 17 points collapse to 0); GNNExplainer mean ≈ 0.50; GNNShap scattered but bounded.

**Interpretation**: PGExplainer is clearly degenerate; GNNExplainer is the most consistent stable explainer overall.

**Related**: A6, A9.

---

## A2 — Spearman by scenario with error bars (money chart)

**Section**: Analysis  
**Type**: Line + error bars  
**Data file**: `analysis/A2_spearman_by_scenario_errorbars.data.csv`

GNNExplainer Spearman mean per scenario, with min–max as error bars and sample counts annotated.

**Key numbers**:
- 1:1 → 0.42 (n=3)
- 1:10 → 0.53 (n=8)
- 1:50 → 0.59 (n=5) — PEAK
- 1:100 → 0.24 (n=1) — COLLAPSE

**Interpretation**: Explainer stability is non-monotonic in imbalance — it climbs until 1:50 then plummets. This is the thesis' central finding.

**Related**: A7, A8, D3.

---

## A3 — Spearman by architecture

**Section**: Analysis  
**Type**: Bar chart  
**Data file**: `analysis/A3_spearman_by_architecture.data.csv`

Mean GNNExplainer Spearman per architecture, sorted descending.

**Key numbers**: GAT = 0.635, TAGCN = 0.590, GCN = 0.484, GraphSAGE = 0.412.

**Interpretation**: Attention (GAT) and topology-aware (TAGCN) architectures yield more stable explanations. Inverse ordering to R2 — confirms the accuracy-vs-stability tension.

**Related**: R2, A4, D2.

---

## A4 — Accuracy vs stability scatter

**Section**: Analysis  
**Type**: Scatter + trend line  
**Data file**: `analysis/A4_accuracy_vs_stability_scatter.data.csv`

Per architecture: mean val F1 on passing configs vs mean GNNExplainer Spearman.

**Interpretation**: Clear negative slope; rank correlation ≈ −0.20. GraphSAGE lives top-right on accuracy but bottom on stability. Shaded band highlights the "stable XAI" region.

**Related**: A3, D2.

---

## A5 — Spearman heatmap (arch × scenario)

**Section**: Analysis  
**Type**: Heatmap  
**Data file**: `analysis/A5_spearman_heatmap_scenario_arch.data.csv`

Per-cell mean GNNExplainer Spearman. Peak cell (1:50 × TAGCN = 0.789) is highlighted with a green border.

**Interpretation**: The diagonal-ish pattern confirms that the peak is jointly produced by arch AND scenario, not one alone.

**Related**: A2, A3, R4.

---

## A6 — Jaccard distribution

**Section**: Analysis  
**Type**: Histogram  
**Data file**: `analysis/A6_jaccard_distribution.data.csv`

All Jaccard values from all explainers, binned.

**Key numbers**: 100% of GNNExplainer + PGExplainer runs (34/34) sit at Jaccard = 1.000.

**Interpretation**: For Jaccard to be informative, the top-k sets must vary across replicas. These explainers are deterministic given the trained model, so Jaccard collapses to 1. Spearman is the only discriminative stability metric for this study.

**Related**: A1, A9.

---

## A7 — Kruskal–Wallis boxplot

**Section**: Analysis  
**Type**: Box plot + statistical test  
**Data file**: `analysis/A7_kruskal_boxplot.data.csv`

Boxplot of GNNExplainer Spearman across the 4 scenarios, with Kruskal–Wallis H-statistic and p-value annotated.

**Key numbers**: H ≈ 4.31, p ≈ 0.23 (not significant at α=0.05 given small n in extreme scenarios).

**Interpretation**: Visual trend (peak-collapse) is strong but statistical power is limited because only n=1 passing config at 1:100. More runs per cell would be needed to confirm.

**Related**: A2, A8.

---

## A8 — Cohen's d effect sizes

**Section**: Analysis  
**Type**: Horizontal bar with magnitude colors  
**Data file**: `analysis/A8_cohens_d_effect_sizes.data.csv`

Pairwise Cohen's d between scenarios (GNNExplainer Spearman).

**Key numbers**: 1:1 vs 1:50 ≈ −0.92 (large), 1:1 vs 1:10 ≈ −0.67 (medium), 1:10 vs 1:50 ≈ −0.35 (small).

**Interpretation**: Effect sizes support the peak claim even though Kruskal–Wallis is n-limited. Larger imbalance (up to 1:50) substantially increases Spearman.

**Related**: A2, A7.

---

## A9 — PGExplainer universal degeneration

**Section**: Analysis  
**Type**: Bar chart (17 zero bars + GNNExplainer reference line)  
**Data file**: `analysis/A9_pgexplainer_degeneration.data.csv`

One bar per PGExplainer run, all at Spearman = 0; horizontal line = GNNExplainer mean.

**Interpretation**: PGExplainer learned a degenerate mask in every config — compare to GNNExplainer's 0.5 baseline. This is not a per-config bug; it is a universal behavior in this setup.

**Related**: A1, A6.

---

## A10 — Top 10 Spearman configs

**Section**: Analysis  
**Type**: Horizontal bar (ranked)  
**Data file**: `analysis/A10_top_spearman_configs.data.csv`

Top 10 configurations ranked by GNNExplainer Spearman, colored by architecture.

**Key numbers**: Peak 1:50_TAGCN_focal_loss = 0.789; runner-up 1:50_GAT_focal_loss = 0.782.

**Interpretation**: Attention-based architectures (GAT, TAGCN) dominate the leaderboard; focal_loss appears in both top slots. A concrete recipe for stable XAI on Elliptic.

**Related**: A3, A5, D4.

---

## D1 — State of the art comparison

**Section**: Discussion  
**Type**: Horizontal bar  
**Data file**: `discussion/D1_sota_comparison.data.csv`

Reported F1 (val or test) for this thesis' best runs vs published baselines.

**Key numbers**:
- Weber 2019 GCN test = 0.41
- Weber 2019 RandomForest test = 0.79
- Pareja 2020 EvolveGCN test = 0.89
- arXiv:2602.23599 GraphSAGE val = 0.85
- This thesis GraphSAGE val best = 0.53
- This thesis GCN val best = 0.31

**Interpretation**: Our static GNNs under-perform temporal architectures. The focus here is stability of XAI on realistic imbalance regimes, not F1 maximisation.

**Related**: R5.

---

## D2 — Accuracy × Stability quadrants

**Section**: Discussion  
**Type**: 2×2 quadrant scatter  
**Data file**: `discussion/D2_accuracy_stability_quadrants.data.csv`

Per-architecture (mean val F1, mean Spearman) plotted with labelled quadrants.

**Interpretation**: No architecture lands in the top-right ("accurate + stable"). Tradeoff regime: pick depending on downstream needs (D4).

**Related**: A3, A4, D4.

---

## D3 — Peak-collapse curve

**Section**: Discussion  
**Type**: Line chart (3 explainers)  
**Data file**: `discussion/D3_peak_collapse_curve.data.csv`

Mean Spearman per scenario for GNNExplainer, GNNShap and PGExplainer overlaid.

**Interpretation**: GNNExplainer and GNNShap share the peak-collapse pattern; PGExplainer is flat at 0. Strong evidence that the peak-collapse phenomenon is intrinsic to the model/imbalance regime, not explainer-specific.

**Related**: A2.

---

## D4 — Architecture recommendation matrix

**Section**: Discussion  
**Type**: Heatmap  
**Data file**: `discussion/D4_recommendation_matrix.data.csv`

Use case × architecture recommendation strength (0 = avoid, 3 = strong).

**Interpretation**: GraphSAGE wins accuracy; GAT/TAGCN win explainability and extreme imbalance. Single actionable summary for practitioners.

**Related**: A3, D2.

---

## D5 — Pipeline runtime breakdown

**Section**: Discussion  
**Type**: Stacked bar  
**Data file**: `discussion/D5_pipeline_runtime_breakdown.data.csv`

Hours spent in Train vs Explain on each machine.

**Key numbers**:
- Machine B (4060): Train 3.1 h, Explain 1.5 h (total 4.6 h)
- Machine C (3050): Train 5.4 h, Explain 24.0 h (total 29.4 h)

**Interpretation**: GAT attention cost dominates explain phase on Machine C. Future work should either shard GAT across more GPUs or restrict to structural explainers.

**Related**: D4.
'''
(OUT / 'FIGURES.md').write_text(figures_md)
print('Wrote', OUT/'FIGURES.md', len(figures_md), 'bytes')

Wrote /tmp/thesis-web-learning/thesis_figures/FIGURES.md 10974 bytes


In [28]:
# Final sanity check
for sub in ('results', 'analysis', 'discussion'):
    pngs = sorted((OUT/sub).glob('*.png'))
    csvs = sorted((OUT/sub).glob('*.data.csv'))
    print(f'{sub}: {len(pngs)} PNGs, {len(csvs)} CSVs')
    for p in pngs: print('  ', p.name)
print('\nDocs:')
for d in ('README.md', 'FIGURES.md'):
    p = OUT / d
    print('  ', d, p.stat().st_size, 'bytes' if p.exists() else 'MISSING')

results: 6 PNGs, 6 CSVs
   R1_pass_rate_by_scenario.png
   R2_pass_rate_by_arch.png
   R3_pass_rate_by_balancing.png
   R4_val_f1_heatmap.png
   R5_val_f1_vs_mcc_scatter.png
   R6_configs_summary_table.png
analysis: 10 PNGs, 10 CSVs
   A10_top_spearman_configs.png
   A1_spearman_distribution_by_explainer.png
   A2_spearman_by_scenario_errorbars.png
   A3_spearman_by_architecture.png
   A4_accuracy_vs_stability_scatter.png
   A5_spearman_heatmap_scenario_arch.png
   A6_jaccard_distribution.png
   A7_kruskal_boxplot.png
   A8_cohens_d_effect_sizes.png
   A9_pgexplainer_degeneration.png
discussion: 5 PNGs, 5 CSVs
   D1_sota_comparison.png
   D2_accuracy_stability_quadrants.png
   D3_peak_collapse_curve.png
   D4_recommendation_matrix.png
   D5_pipeline_runtime_breakdown.png

Docs:
   README.md 2946 bytes
   FIGURES.md 11110 bytes
